# Identifiers

This notebook connects the de-identified dicoms in ECS to the HeartLab report database. <br>
The resulting mappings are saved to `patient_to_study.csv`.

In [10]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

In [11]:
deid = pd.read_csv('dataset/echo-study-deid.csv')
deid = deid.sort_values(by="OriginalPatientID", ascending=True)

numeric_column = pd.to_numeric(deid["OriginalPatientID"], errors='coerce')
filtered_df = deid[numeric_column.notna()]

deid = filtered_df
deid.head()

,OriginalStudyID,OriginalPatientID,DeidentifiedStudyID,DeidentifiedPatientID
96930,1.2.124.113532.36.59504.16340.20110317.142116.7522070,2393668,1.2.276.0.7230010.3.1.2.845494328.1.1703200559.14758115,aa70416a-8f28-4e88-ac03-be28166d5dd0
75793,1.2.840.113680.1.103.51602.1120748752.341366,3233860,1.2.276.0.7230010.3.1.2.811753780.1.1703538509.15238312,0d9932d3-a73f-40c3-b994-9d9889b5ebef
155940,1.2.840.113680.1.103.51602.1137682790.772181,3278818,1.2.276.0.7230010.3.1.2.845494328.1.1703543619.20977646,225755fc-cc1c-4b5c-83f3-1fe3615d4c0d
192671,1.2.840.113680.1.103.51602.1209402194.673271,3697059,1.2.276.0.7230010.3.1.2.1714578744.1.1703561546.15322813,e4f9820f-c1c3-4ac0-b8d0-1acc17899b9b
101400,1.2.840.113680.1.103.52717.1132852745.214089,6043967,1.2.276.0.7230010.3.1.2.859333938.1.1703571621.15565912,33b75629-5230-4c8e-8701-ffa46e5577d8


In [12]:
studies = pd.read_csv('dataset/STUDIES.csv')
studies = studies.sort_values(by="PATI_ID", ascending=False)
studies.head()

/tmp/ipykernel_120476/3782090118.py:1: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  studies = pd.read_csv('dataset/STUDIES.csv')


,ID,PATI_ID,ACCESSION_NUMBER,REFERRING_PHYSICIANS_NAME,STUDY_DATE,STUDY_DESCRIPTION,STUDY_ID,STUDY_INSTANCE_UID,STUDY_TIME,STUDY_STATE,STUDY_SOURCE,ADM_ID,DICTATIONS_DATE,DICATATION_DATE,DICTATION_DATE,FLAG,RECONCILED_DATE
431534,468624,597091,34157427,NaN,10/11/16 09:12:07,Transesophageal Echocardiogram,NaN,1.3.12.2.1107.5.8.9.1002655211149138.20161011131345756,91207.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
431667,468623,597081,333333333,NaN,09/04/15 13:43:32,TAVI,NaN,1.2.392.200036.9116.3.1.22002075.4.0.7638948729961748,134332.0,NaN,NaN,NaN,NaN,NaN,NaN,43.0,NaN
431579,468622,597080,A201501080820391,NaN,01/08/15 11:02:53,Cardiac,NaN,1.3.46.670589.28.26540784710020150108160251267561,110253.0,NaN,NaN,NaN,NaN,NaN,NaN,19.0,NaN
431666,468621,597079,1811141SMGH,NaN,08/12/15 08:07:09,CARDIAC CATH INVESTIGATION,NaN,1.2.840.113845.11.1000000002075521234.20150812064903.7362852,80709.0,NaN,NaN,NaN,NaN,NaN,NaN,37.0,NaN
431665,468620,597078,0000000,NaN,09/04/15 18:03:36,EVAR,NaN,1.2.392.200036.9116.3.1.22002075.4.0.7638950419662720,180336.0,NaN,NaN,NaN,NaN,NaN,NaN,73.0,NaN


In [13]:
series = pd.read_csv('dataset/SERIES.csv')
series.head()

/tmp/ipykernel_120476/2707617270.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  series = pd.read_csv('dataset/SERIES.csv')


,ID,EQUI_ID,STUD_ID,ARCHIVE_STATUS,BODY_PART_EXAMINED,ICON_FILENAME,MODALITY,PERFORMING_PHYSICIANS_NAME,LATERALITY,LAST_READ_DATE,SERIES_DATE,SERIES_DESCRIPTION,SERIES_INSTANCE_UID,SERIES_NUMBER,SERIES_TIME,PATIENT_POSITION,PROTOCOL_NAME,DEPARTMENT_ID
0,442524,154.0,227777,U,NaN,NaN,US,NaN,NaN,NaN,NaN,NaN,1.2.840.113680.1.103.60427.1265116737.713938.1,1.0,NaN,NaN,NaN,NaN
1,442525,1.0,227774,U,NaN,NaN,SR,NaN,NaN,NaN,NaN,NaN,1.2.840.113680.1.103.57589.1265126889.672993.2.2,NaN,NaN,NaN,NaN,NaN
2,442537,NaN,227784,U,NaN,NaN,SR,NaN,NaN,NaN,NaN,NaN,IMAGELESS.SERIES.UID.442537,NaN,NaN,NaN,NaN,NaN
3,442539,NaN,227786,U,NaN,NaN,SR,NaN,NaN,NaN,NaN,NaN,IMAGELESS.SERIES.UID.442539,NaN,NaN,NaN,NaN,NaN
4,442540,NaN,227787,U,NaN,NaN,SR,NaN,NaN,NaN,NaN,NaN,IMAGELESS.SERIES.UID.442540,NaN,NaN,NaN,NaN,NaN


In [14]:
overlap_count = deid['OriginalStudyID'].isin(studies['STUDY_INSTANCE_UID']).sum()
overlap_count

223450

In [15]:
# Merge A and B to align study IDs and include all patient IDs
studies_renamed = studies.rename(columns={'STUDY_INSTANCE_UID': 'OriginalStudyID'})
merged_df = pd.merge(deid, studies_renamed, on='OriginalStudyID', how='left')

# Group by patient ID and aggregate study IDs into a list
result_df = merged_df.groupby('DeidentifiedPatientID').agg({
    'OriginalPatientID': lambda x: x,
    'DeidentifiedStudyID': lambda x: list(x),
    'OriginalStudyID': lambda x: list(x),
}).reset_index()

# Rename columns for clarity if needed
result_df.columns = ['deidentified_patient_id', 'original_patient_id', 'deidentified_study_ids', 'original_study_ids']

In [16]:
result_df.head()

,deidentified_patient_id,original_patient_id,deidentified_study_ids,original_study_ids
0,00001302-70bf-4ec4-8bde-1b3f9bf98689,0595291,[1.2.276.0.7230010.3.1.2.1714512485.1.1703546444.18496766],[1.2.840.113680.1.103.51602.1144435549.722003]
1,0000431a-2b47-4042-9f2b-1da11dd94331,3355179,[1.2.276.0.7230010.3.1.2.1714512485.1.1703485133.17708074],[1.2.840.113619.2.98.8078.1177657558.0.1371]
2,000058d1-b49e-4830-8579-772ad15ab48b,2149134,[1.2.276.0.7230010.3.1.2.1714578744.1.1703255110.10341847],[1.2.124.113532.36.59504.16340.20121128.132749.16537349]
3,00007991-bbc2-4606-98a9-473a96ce38c1,3058413,[1.2.276.0.7230010.3.1.2.859333938.1.1703242935.10155698],[1.2.124.113532.36.59504.16340.20120719.125804.5993639]
4,00008cd9-36ef-49e3-bf35-0d8f01cf77af,3930997,[1.2.276.0.7230010.3.1.2.1714578744.1.1703156371.8603683],[1.2.124.113532.30763.51996.30715.20141205.135828.58140391]


In [17]:
average_length = result_df['original_study_ids'].apply(len).mean()
print(f"{len(result_df):,} patients with studies @ {average_length} per patient")

232,848 patients with studies @ 1.000060125060125 per patient


In [21]:
result_df.to_csv('patient_to_study.csv')

In [18]:
# metadata/
# 1.2.276.0.7230010.3.1.2.1714512485.1.1703113053.10353948_
# 1.2.276.0.7230010.3.1.3.1714512485.1.1703114855.10373069_
# 1.2.276.0.7230010.3.1.4.1714512485.1.1703114862.10373184
# .dcm.txt

In [19]:
specific_string = '1.2.276.0.7230010.3.1.2.1714512485.1.1703113053.10353948'

filtered_df = result_df[result_df['deidentified_study_ids'].apply(lambda lst: specific_string in lst)]
filtered_df.head()

,deidentified_patient_id,original_patient_id,deidentified_study_ids,original_study_ids
43544,2fdfd227-48b5-4d34-aa83-3e0230124300,4109865,[1.2.276.0.7230010.3.1.2.1714512485.1.1703113053.10353948],[1.2.124.113532.30763.51996.30715.20140312.80222.545351]


### Notes
1. Link the parent dicom directory names (metadata df) to the deidentified study ids.
2. Look up the original study id by deidentified study id.
3. Reference the original study id within the database.
4. Use original study id to look up report id
5. Use report id to look up measurements
6. Use report id to look up exam demographics
7. Use report id to look up findings

### Metadata
We can also link the deidentified study ids to the metadata files.

In [4]:
import os
import pandas as pd

# Directory containing the metadata files
directory = 'metadata'

# List to store the file info
file_info = []

# Iterate over files in the specified directory
for filename in os.listdir(directory):
    if filename.endswith('.txt'):  # Check if the file is a text file
        full_path = os.path.join(directory, filename)
        parts = filename.split('_')  # Split the filename by underscore
        
        # Check if there are three parts: parent, subdir, dicom
        if len(parts) == 3:
            # Append the information to the list
            file_info.append({
                'full_filename': filename,
                'parent': parts[0],
                'subdir': parts[1],
                'dicom': parts[2].split('.dcm')[0]  # Remove the .dcm.txt part from the last section
            })

# Create a DataFrame from the list
metadata = pd.DataFrame(file_info)
metadata.head()

,full_filename,parent,subdir,dicom
0,1.2.276.0.7230010.3.1.2.1714512485.1.170311302...,1.2.276.0.7230010.3.1.2.1714512485.1.170311302...,1.2.276.0.7230010.3.1.3.1714512485.1.170311302...,1.2.276.0.7230010.3.1.4.1714578744.1.170311336...
1,1.2.276.0.7230010.3.1.2.1714512485.1.170311304...,1.2.276.0.7230010.3.1.2.1714512485.1.170311304...,1.2.276.0.7230010.3.1.3.1714512485.1.170311304...,1.2.276.0.7230010.3.1.4.859333938.1.1703113132...
2,1.2.276.0.7230010.3.1.2.1714512485.1.170311432...,1.2.276.0.7230010.3.1.2.1714512485.1.170311432...,1.2.276.0.7230010.3.1.3.1714512485.1.170311432...,1.2.276.0.7230010.3.1.4.845494328.1.1703114543...
3,1.2.276.0.7230010.3.1.2.1714512485.1.170311453...,1.2.276.0.7230010.3.1.2.1714512485.1.170311453...,1.2.276.0.7230010.3.1.3.1714512485.1.170311453...,1.2.276.0.7230010.3.1.4.859333938.1.1703115025...
4,1.2.276.0.7230010.3.1.2.1714512485.1.170311307...,1.2.276.0.7230010.3.1.2.1714512485.1.170311307...,1.2.276.0.7230010.3.1.3.1714578744.1.170311342...,1.2.276.0.7230010.3.1.4.845494328.1.1703114074...


In [5]:
metadata.to_csv('metadata.csv')

In [6]:
len(metadata)

9970

In [20]:
specific_string = metadata.iloc[0]['parent']

filtered_df = result_df[result_df['deidentified_study_ids'].apply(lambda lst: specific_string in lst)]
filtered_df.head()

,deidentified_patient_id,original_patient_id,deidentified_study_ids,original_study_ids
32627,23d8b044-e0d6-45b7-9eac-dc98b4077a98,4065751,[1.2.276.0.7230010.3.1.2.1714512485.1.1703113026.10353697],[1.2.124.113532.17.9614.32792.20120813.90215.16006696]


In [23]:
file_path = 'metadata/' + metadata.iloc[0]['full_filename']

# Open the file in read mode ('r')
with open(file_path, 'r') as file:
    # Read the content of the file
    file_content = file.read()
    
    # Print the content of the file
    print(file_content)


# Dicom-File-Format

# Dicom-Meta-Information-Header
# Used TransferSyntax: Little Endian Explicit
(0002,0000) UL 238                                      #   4, 1 FileMetaInformationGroupLength
(0002,0001) OB 00\01                                    #   2, 1 FileMetaInformationVersion
(0002,0002) UI =UltrasoundMultiframeImageStorage        #  28, 1 MediaStorageSOPClassUID
(0002,0003) UI [1.2.276.0.7230010.3.1.4.1714578744.1.1703113364.7967611] #  56, 1 MediaStorageSOPInstanceUID
(0002,0010) UI =JPEGLSLossless                          #  22, 1 TransferSyntaxUID
(0002,0012) UI [1.2.826.0.1.3680043.2.1143.107.104.103.115.3.0.21] #  50, 1 ImplementationClassUID
(0002,0013) SH [GDCM 3.0.21]                            #  12, 1 ImplementationVersionName
(0002,0016) AE [gdcmconv]                               #   8, 1 SourceApplicationEntityTitle

# Dicom-Data-Set
# Used TransferSyntax: JPEG-LS Lossless
(0008,0005) CS [ISO_IR 100]                             #  10, 1 SpecificCharacterSet
(00